## Setup: Install and Import

In [19]:
!pip install --upgrade sentence-transformers huggingface_hub --break-system-packages

import os
import pandas as pd
import nltk
from scipy import spatial
from sentence_transformers import SentenceTransformer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # Add this line
nltk.download('averaged_perceptron_tagger', quiet=True)

print("✓ Setup complete!")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✓ Setup complete!


## Configuration

In [ ]:

MITRE_CSV = "/home/jovyan/work/MITREMapping/enterprise-techniques.csv"          # MITRE ATT&CK CSV
OUTPUT_FILE = 'mapped_patterns.csv'              # Output file

THRESHOLD = 0.9        # Distance threshold (paper optimal: 0.6)
WEIGHT_TITLE = 0.5     # Title weight (paper optimal: 0.4)

# =============================

## Load Functions

In [40]:
# Initialize
bert_model = SentenceTransformer('all-mpnet-base-v2')
embedding_memo = {}

# Load MITRE framework
def load_attack_patterns(csv_path):
    df = pd.read_csv(csv_path)
    attack_pattern_dict = {}
    prev_id = None
    
    for _, row in df.iterrows():
        _id = row['ID']
        if not pd.isnull(_id):
            attack_pattern_dict[_id] = [[row['Name'], row['Description']]]
            prev_id = _id
        else:
            if prev_id:
                attack_pattern_dict[prev_id].append([row['Name'], row['Description']])
    
    return attack_pattern_dict

# Embedding functions
def get_embedding(txt):
    if txt in embedding_memo:
        return embedding_memo[txt]
    emb = bert_model.encode([txt])[0]
    embedding_memo[txt] = emb
    return emb

def get_embedding_distance(txt1, txt2):
    p1 = get_embedding(txt1)
    p2 = get_embedding(txt2)
    return spatial.distance.cosine(p1, p2)

# Mapping function
def get_mitre_id(text, attack_pattern_dict, weight_title):
    min_dist = 25
    ret = None
    
    for k, tech_list in attack_pattern_dict.items():
        for v in tech_list:
            d = (weight_title * get_embedding_distance(text, v[0]) + 
                 (1 - weight_title) * get_embedding_distance(text, v[1]))
            if d < min_dist:
                min_dist = d
                ret = k
    
    return ret, min_dist

# Text processing
def remove_consec_newline(s):
    if not s:
        return s
    ret = s[0]
    for x in s[1:]:
        if not (x == ret[-1] and ret[-1] == '\n'):
            ret += x
    return ret

def extract_sentences(text):
    text = remove_consec_newline(text)
    text = text.replace('\t', ' ').replace("\'", "'")
    sents_nltk = nltk.sent_tokenize(text)
    sents = []
    for x in sents_nltk:
        sents += x.split('\n')
    return [s.strip() for s in sents if s.strip()]

print("✓ Functions loaded")

Loading weights: 100%|██████████████████████████████████████████████| 199/199 [00:00<00:00, 579.74it/s, Materializing param=pooler.dense.weight]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Functions loaded


## Run Mapping

In [41]:
print("Loading MITRE ATT&CK framework...")
attack_pattern_dict = load_attack_patterns(MITRE_CSV)
print(f"✓ Loaded {len(attack_pattern_dict)} techniques\n")

# Read input file
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    text = f.read()

# -----------------------------
# 1️⃣ Load Ground Truth
# -----------------------------
gt = pd.read_csv("/home/jovyan/work/MITREMapping/mapping-gt-250-2.csv")

print("Ground truth samples:", len(gt))
print("GT Columns:", gt.columns.tolist())

TRUE_COL = "true_parent"
PHRASE_COL = "phrase"

print("THRESHOLD")
print(THRESHOLD)

print("WEIGHT_TITLE")
print(WEIGHT_TITLE)

mapped = {}
correct_matches = 0
total_predictions = 0

for i, row in gt.iterrows():

    phrase = row[PHRASE_COL]
    true_parent = row[TRUE_COL]

    _id, dist = get_mitre_id(phrase, attack_pattern_dict, WEIGHT_TITLE)

    if dist < THRESHOLD:
        total_predictions += 1

        # Store best example per technique (your original logic)
        if _id not in mapped or dist < mapped[_id][0]:
            mapped[_id] = (dist, phrase)

        # Check correctness
        is_correct = (_id == true_parent)
        if is_correct:
            correct_matches += 1

        status = "✓" if is_correct else "✗"

        print(
            f"[{i:2d}] {status} "
            f"TRUE: {true_parent} | "
            f"PRED: {_id} | "
            f"Dist: {dist:.3f} | "
            f"{phrase[:50]}..."
        )

    else:
        print(
            f"[{i:2d}] - BELOW THRESHOLD | "
            f"TRUE: {true_parent} | "
            f"BestDist: {dist:.3f} | "
            f"{phrase[:50]}..."
        )

print("\n==============================")
print(f"✓ Mapped {len(mapped)} unique techniques")
print(f"✓ Total Predictions: {total_predictions}")
print(f"✓ Correct Matches: {correct_matches}")

if total_predictions > 0:
    accuracy = correct_matches / total_predictions
    print(f"✓ Accuracy (on predicted only): {accuracy:.4f}")
else:
    print("No predictions above threshold.")

Loading MITRE ATT&CK framework...
✓ Loaded 191 techniques

Ground truth samples: 250
GT Columns: ['phrase_id', 'phrase', 'true_parent', 'true_subtech']
THRESHOLD
0.9
WEIGHT_TITLE
0.5
[ 0] ✓ TRUE: T1055 | PRED: T1055 | Dist: 0.503 | Analysis of the implant's behaviour revealed that ...
[ 1] ✗ TRUE: T1055 | PRED: T1574 | Dist: 0.433 | The threat actor staged a malicious DLL in a world...
[ 2] ✗ TRUE: T1055 | PRED: T1547 | Dist: 0.517 | Telemetry captured CreateRemoteThread calls preced...
[ 3] ✗ TRUE: T1055 | PRED: T1547 | Dist: 0.437 | Post-exploitation activity included classic LoadLi...
[ 4] ✗ TRUE: T1055 | PRED: T1574 | Dist: 0.473 | The backdoor achieved code execution within lsass....
[ 5] ✓ TRUE: T1055 | PRED: T1055 | Dist: 0.428 | A kernel driver obtained via a BYOVD technique was...
[ 6] ✗ TRUE: T1055 | PRED: T1574 | Dist: 0.403 | A trojanised version of a legitimate DLL was place...
[ 7] ✗ TRUE: T1055 | PRED: T1574 | Dist: 0.477 | The loader persisted its core library by encodi

## Save and Display Results

In [42]:
import pandas as pd

# =====================================================
# 🔎 Top-K Retrieval Function
# =====================================================
def get_top_k_mitre_ids(text, attack_pattern_dict, weight_title, k=5):
    results = []

    for parent_id, tech_list in attack_pattern_dict.items():
        for v in tech_list:

            title = v[0]
            description = v[1]

            dist_title = get_embedding_distance(text, title)
            dist_desc = get_embedding_distance(text, description)

            dist = weight_title * dist_title + (1 - weight_title) * dist_desc

            results.append((parent_id, dist))

    # Sort by distance (ascending)
    results = sorted(results, key=lambda x: x[1])

    # Remove duplicate technique IDs (keep best distance)
    unique = {}
    for tech_id, dist in results:
        if tech_id not in unique:
            unique[tech_id] = dist

    return list(unique.items())[:k]


# =====================================================
# 📊 Load Ground Truth
# =====================================================
gt = pd.read_csv("/home/jovyan/work/MITREMapping/mapping-gt-250-2.csv")

TRUE_COL = "true_parent"
PHRASE_COL = "phrase"

print("THRESHOLD:", THRESHOLD)
print("WEIGHT_TITLE:", WEIGHT_TITLE)

rows = []

for _, row in gt.iterrows():

    phrase = row[PHRASE_COL]
    true_parent = row[TRUE_COL]

    top_matches = get_top_k_mitre_ids(
        phrase,
        attack_pattern_dict,
        WEIGHT_TITLE,
        k=5
    )

    top_ids = [tech_id for tech_id, _ in top_matches]

    rows.append({
        "TRUE_PARENT": true_parent,
        "TOP_1": top_ids[0],
        "TOP_2": top_ids[1],
        "TOP_3": top_ids[2],
        "TOP_4": top_ids[3],
        "TOP_5": top_ids[4],
    })

results_df = pd.DataFrame(rows)


# =====================================================
# 🎨 Highlight Correct Prediction in Red
# =====================================================
def highlight_correct(row):
    styles = []
    true_val = row["TRUE_PARENT"]

    for col in row.index:
        if col.startswith("TOP_") and row[col] == true_val:
            styles.append("color: red; font-weight: bold")
        else:
            styles.append("")
    return styles


styled_df = results_df.style.apply(highlight_correct, axis=1)

styled_df

THRESHOLD: 0.9
WEIGHT_TITLE: 0.5


,TRUE_PARENT,TOP_1,TOP_2,TOP_3,TOP_4,TOP_5
0,T1055,T1055,T1574,T1546,T1218,T1547
1,T1055,T1574,T1055,T1546,T1218,T1216
2,T1055,T1547,T1055,T1562,T1546,T1543
3,T1055,T1547,T1574,T1055,T1546,T1218
4,T1055,T1574,T1547,T1055,T1003,T1620
5,T1055,T1055,T1546,T1574,T1218,T1547
6,T1055,T1574,T1546,T1218,T1055,T1036
7,T1055,T1574,T1620,T1055,T1546,T1129
8,T1055,T1574,T1055,T1127,T1553,T1195
9,T1055,T1574,T1055,T1620,T1546,T1129


In [ ]:
N = len(results_df)

rank_counts = {1:0, 2:0, 3:0, 4:0, 5:0}

for _, row in results_df.iterrows():
    true_label = row["TRUE_PARENT"]
    
    for r in range(1, 6):
        if row[f"TOP_{r}"] == true_label:
            rank_counts[r] += 1
            break   # stop at first occurrence

# Convert to percentages
rank_percent = {r: (rank_counts[r] / N) * 100 for r in rank_counts}

# Cumulative Top-K
cumulative = {}
running = 0
for r in range(1, 6):
    running += rank_counts[r]
    cumulative[r] = (running / N) * 100

# Display table
eval_df = pd.DataFrame({
    "Rank": list(rank_percent.keys()),
    "% at Exact Rank": list(rank_percent.values()),
    "Cumulative Top-K %": list(cumulative.values())
})

display(eval_df)

# Exact rank %
for r in range(1, 6):
    summary_row[f"Rank_{r}_%"] = rank_percent[r]

# Cumulative Top-K %
for r in range(1, 6):
    summary_row[f"Top_{r}_%"] = cumulative[r]

# Convert to DataFrame (single row)
eval_summary_df = pd.DataFrame([summary_row])

display(eval_summary_df)


,Rank,% at Exact Rank,Cumulative Top-K %
0,1,45.2,45.2
1,2,20.4,65.6
2,3,8.0,73.6
3,4,2.4,76.0
4,5,6.0,82.0


,Weight,Threshold,Rank_1_%,Rank_2_%,Rank_3_%,Rank_4_%,Rank_5_%,Top_1_%,Top_2_%,Top_3_%,Top_4_%,Top_5_%
0,0.5,0.9,45.2,20.4,8.0,2.4,6.0,45.2,65.6,73.6,76.0,82.0
